In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

In [2]:
# generate PTDF matrices and capacity vectors for normal ops and 2 outage scenarios

M = np.array([[1,1,0,0],
            [0,0,1,0],
            [0,0,0,0]])

buses = ['A','B', 'C', 'D']

lines = [
    ('A','B'),
    ('B','C'),
    ('C','D'),
    ('A','D'),
    ('B','D'),
]

K = np.zeros((len(lines), len(buses)) )

for i, (start, stop) in enumerate(lines):
    K[i, buses.index(start)] = 1
    K[i, buses.index(stop)] = -1


x = np.ones(5)

capacities = np.array([10, 9, 11, 10, 6]) # line capacities

costs = np.array([20, 35, 50, 90]) # generator costs

d = np.array([0,0,5]) # reduced demand vector

contingencies = [
    ("B", "D"),
    ("B", "C")
]


# calculate PTDF matrix given a list of active lines
def calculate_ptdf(active_lines):
    
    K = np.zeros((len(active_lines), len(buses)) )

    for i, (start, stop) in enumerate(active_lines):
        K[i, buses.index(start)] = 1
        K[i, buses.index(stop)] = -1
    
    B = K.T @ K

    B_red = B[:(len(buses)-1), :(len(buses)-1)] # remove reference BUS row/ col

    thetas = np.linalg.solve(B_red, np.eye(len(buses)-1))
    thetas_full = np.vstack( (thetas, np.zeros(len(buses)-1)) )

    return( K @ thetas_full)

normal_ptdf = calculate_ptdf(lines)

bd_ptdf = calculate_ptdf(
    [line for line in lines if line != ("B", "D")]
)

bc_ptdf = calculate_ptdf(
    [line for line in lines if line != ("B", "C")]
)

capacities_bd = np.delete(capacities, lines.index(('B', 'D')))
capacities_bc = np.delete(capacities, lines.index(('B', 'C')))

A_list = []
b_list = []

ptdfs = [normal_ptdf, bd_ptdf, bc_ptdf]
caps = [capacities, capacities_bd, capacities_bc]

for ptdf, cap in zip(ptdfs, caps):
    # ptdf is a 2D matrix, -ptdf is a 2D matrix
    A_list.append(ptdf @ M)
    A_list.append(-ptdf @ M)
    
    # caps is a 1D array
    b_list.append(cap + ptdf @ d)
    b_list.append(cap - ptdf @ d)


A_ub = np.vstack(A_list) 
b_ub = np.concatenate(b_list)


In [3]:
base_lines = ["AB", "BC", "CD", "AD", "BD"]

lines_normal = base_lines[:] # Copy all 5
lines_bd = base_lines[:]
del lines_bd[lines.index(('B', 'D'))] 

lines_bc = base_lines[:]
del lines_bc[lines.index(('B', 'C'))] 

scenario = ["Normal", "B-D Outage", "B-C Outage"]
line_list = [lines_normal, lines_bd, lines_bc]

constraint_data = []

for scenarios, active_line in zip(scenario, line_list):
    for line in active_line:
        constraint_data.append({
            "Scenario": scenarios,
            "Monitored Line": line,
            "Direction": "Positive"
        })
    for line in active_line:
        constraint_data.append({
            "Scenario": scenarios,
            "Monitored Line": line,
            "Direction": "Negative"
        })


#zip(scenario_prefixes, scenario_line_lists)

constraints = pd.DataFrame(constraint_data)

In [4]:
# Energy balance: g_A + b_B + g_C = 16
A_eq = [[1, 1, 1, 1]]
b_eq = [16]
bounds = [
    (0, 10),  # G1
    (0, 4),   # G2
    (0, 12),  # G3
    (0, 16)   # G4
]

print(b_ub[13])

result = linprog(
    c=costs,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)


constraints["Shadow Price"] = -result.ineqlin.marginals
constraints["Slack"] = result.ineqlin.residual

binding = constraints.loc[np.isclose(constraints['Slack'],0),]
print(binding)

print("Optimal dispatch:", result.x)
print("Total cost:", result.fun)
print(result.eqlin.marginals) # why does this imply a marginal MW at D? 90

# LMP at A
mu = constraints.loc[(constraints["Scenario"] == 'B-D Outage') & 
                (constraints["Monitored Line"] == 'AD') &
                (constraints["Direction"] == 'Positive')
                , "Shadow Price"]

# print(f"LMP A: {90 - .75 * 73.3333}")
# print(f"LMP B: {90 - .5 * 73.3333}")
# print(f"LMP C: {90 - .25 * 73.3333}")

# bd_ptdf

11.25
      Scenario Monitored Line Direction  Shadow Price  Slack
13  B-D Outage             AD  Positive          70.0    0.0
21  B-C Outage             BD  Positive           7.5    0.0
Optimal dispatch: [10.    3.5   2.25  0.25]
Total cost: 457.5
[90.]


In [5]:
#result.x

pt = np.array([[1, 1, 1],  [1/3, 2/3, 0], [3/4, 1/2, 0] ])
deltas = np.linalg.solve(pt, np.array([0, .01, 0]))

(deltas @ costs[1:4])
#A_ub[21,:]

np.float64(-0.07499999999999984)

In [6]:

# Identify binding transmission constraints
active_lines = np.where(
    np.isclose(result.ineqlin.residual, 0, atol=1e-8)
)[0]

# Identify generators at their maximum
active_upper = np.where(
    np.isclose(result.upper.residual, 0, atol=1e-8)
)[0]

# Identify generators at their min
active_lower = np.where(
    np.isclose(result.lower.residual, 0, atol=1e-8)
)[0]


print("Active transmission rows:", active_lines)
print("Generators at upper bounds:", active_upper)
print("Generators at lower bounds:", active_lower)


n_gen = len(costs)

P = np.vstack([
    np.ones(n_gen),          # energy balance
    A_ub[active_lines],     # binding transmission constraints
    np.eye(n_gen)[active_upper],  # generator upper bounds
    -np.eye(n_gen)[active_lower]
])

print(P)


y = np.linalg.solve(P.T, costs)

labels = (
    ["Energy balance"]
    + [f"Line constraint {i}" for i in active_lines]
    + [f"G{i+1} upper bound" for i in active_upper]
    + [f"G{i+1} lower bound" for i in active_lower]
)

print(pd.Series(y, index=labels))




Active transmission rows: [13 21]
Generators at upper bounds: [0]
Generators at lower bounds: []
[[1.         1.         1.         1.        ]
 [0.75       0.75       0.5        0.        ]
 [0.33333333 0.33333333 0.66666667 0.        ]
 [1.         0.         0.         0.        ]]
Energy balance        90.0
Line constraint 13   -70.0
Line constraint 21    -7.5
G1 upper bound       -15.0
dtype: float64


In [7]:
def gen_new_vectorized(delta):
    # Ensure delta is a numpy array
    delta = np.asarray(delta)
    
    # Broadcast the constant 10 to match the shape of the delta array
    g1 = np.full_like(delta, 10.0) 
    g2 = 3.5 + 2 * delta
    g3 = 2.25 - delta
    g4 = 0.25 - delta
    
    # Combine into a 2D array where each row represents one delta
    return np.column_stack((g1, g2, g3, g4))

# Generate a sequence of values (e.g., 0.01 to 2.0 with 200 steps)
deltas = np.linspace(0.01, 2.0, num=200) 

# Run the function once on the entire sequence
results = gen_new_vectorized(deltas)

df = pd.DataFrame(results, columns=['g1', 'g2', 'g3', 'g4'])

# 2. Assign the 1D deltas array as a new column
df['delta'] = deltas

In [35]:
ptdf_single = np.vstack(ptdfs)

#print(ptdf_single[:5,])
#pdtdf_single[12,]

(ptdf_single[:5,] @ np.array([0,0,.01]))

array([-0.00125, -0.00375,  0.00625,  0.00125,  0.0025 ])

In [85]:

#bd_ptdf[3,] @ np.array([0,0,.01]) # A–D after B–D outage
#bc_ptdf[3,] @ np.array([0,0,.01]) # B–D after B–C outage
#delta_b = np.array([0.01, 0.0025, 0 , 0])

#delta_g = np.linalg.solve(P, delta_b)
#costs @ delta_g
#delta_b

def get_lmp(bus):
    empty = np.zeros(3)
    if bus == 'D': 
        delta_b = np.array([.01,0,0,0])
    else: 
        empty[buses.index(bus)] = .01
        delta_b = np.array([.01, 
                                bd_ptdf[3,] @ empty,
                                bc_ptdf[3,] @ empty,
                                0])
    delta_g = np.linalg.solve(P, delta_b)
    return(costs @ delta_g * 100)

get_lmp('D')

for bus in buses:
    print(bus, get_lmp(bus))

A 35.00000000000001
B 50.0
C 72.5
D 90.0
